# Decoupled RoPE in MLA

This notebook activates the positional query/key path and runs the complete naive MLA forward pass.

In [1]:
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))
torch.manual_seed(7)

## Inspect the two score contributions

Only the positional query and key features receive RoPE.

In [2]:
from src.model import DeepSeekConfig
from src.model.mla import KVCompression, QueryCompression
from src.model.rope import RotaryEmbedding

config = DeepSeekConfig()
x = torch.randn(2, 16, config.dim)
q_content, q_position = QueryCompression(config)(x)
_, k_position, k_content, values = KVCompression(config)(x)

q_content = q_content.transpose(1, 2)
q_position = q_position.transpose(1, 2)
k_content = k_content.transpose(1, 2)
k_position = k_position.transpose(1, 2).unsqueeze(1).expand(-1, config.n_heads, -1, -1)
rope = RotaryEmbedding(config.qk_rope_head_dim, config.max_seq_len)
q_position = rope(q_position)
k_position = rope(k_position)

content_scores = q_content @ k_content.transpose(-2, -1)
position_scores = q_position @ k_position.transpose(-2, -1)
print("content scores:", content_scores.shape)
print("position scores:", position_scores.shape)

content scores: torch.Size([2, 8, 16, 16])
position scores: torch.Size([2, 8, 16, 16])


## Run complete naive MLA

The output keeps the hidden dimension, so the layer can later replace standard attention inside a Transformer block.

In [3]:
from src.model.mla import MLA

model = MLA(config)
output = model(x)
print("input:", x.shape)
print("output:", output.shape)

output.square().mean().backward()
assert model.q_branch.q_down.weight.grad is not None
assert model.kv_branch.kv_down.weight.grad is not None
print("MLA backward check passed")

input: torch.Size([2, 16, 256])
output: torch.Size([2, 16, 256])
MLA backward check passed
